In [1]:
import numpy as np
from scipy import stats

# Sample Size Selection - Continuous Data

Statistical power is an important concept in statistical anaysis. Traditionally it is the measure of rejecting $H_0$ under $H_1$:

$\text{Power} = P(\text{Reject } H_0 | H_1)$ 

Using statistical tests with low sample sizes can be misleading and the probability of the result being wrong is higher. On the other hand, using a larger sample is more costly hence not feasible. Therefore, determining a proper sample size aligning with the expectations is important. In this notebook we will see different ways to determine appropriate sample size.

## Sample Size Selection Based on Standard Error

The goal in this method is to keep a certain precision on the predicted statistic. As we saw before, a two-sided interval is defined with error rate $\alpha$ as

$CI = \bar{x} \pm t_{\alpha/2,\nu} \times SE$

Under large $n$, this equation converges to

$CI = \bar{x} \pm z_{\alpha/2} \cdot SE$

We know that

$SE = \frac {\sigma} {\sqrt{n}}$

So everything being equal, the more we want to keep the confidence interval narrower, the more samples we need to collect. 

The $\sigma$ values can be gathered from external resources directly or derived from confidence intervals reported.


**Example**

Suppose we are conducting a study on a drug used to treat hyperglycemia, and we want to be precise about the degree of blood sugar reduction it causes. Specifically, we want an interval that deviates by at most 10 mg/dL from the population mean.

Based on previous studies, we know that $\sigma = 30$. Then, for a 5% error rate:

$E = z_{\alpha/2} \cdot SE$

$= z_{\alpha/2} \cdot \frac {\sigma} {\sqrt{n}} = 10$

$= 1.96 \cdot \frac {30} {\sqrt{n}} = 10$

In [2]:
alpha = 0.05
z = stats.norm.ppf(1 - alpha/2)
sigma = 30
E = 10

n = (z * sigma / E) ** 2
n = np.ceil(n) 

print(f"z-critical: {z:.4f}")
print(f"Required sample size: {int(n)}")

z-critical: 1.9600
Required sample size: 35


Alternatively, we can use Stein's method for sequential experiment design. This is especially useful if we have no prior information on $\sigma$. 

### Stein's Method for Sequential Experiment Design

In Stein's method, first a pilot sample is collected. The pilot sample size, sample standard deviation, and critical value are computed and used in the second step. To explain more precisely, the total sample size required to achieve a confidence interval with $2d$ width is calculated using:

$N = \max\left(n_0,\ \left\lfloor \left(\dfrac{t_{n_0-1,\,\alpha/2}\cdot s_0}{d}\right)^2 \right\rfloor + 1\right)$

where $n_0$ and $s_0$ are the sample size and standard deviation of the pilot sample, respectively.

Then collect $N - n_0$ additional samples, and compute the confidence interval using the critical value and standard deviation from the pilot sample together with the overall sample mean:

$CI = \bar{x}_N \pm t_{n_0-1,\,\alpha/2}\cdot \frac{s_0}{\sqrt{N}}$

where $\bar{x}_N$ is the mean of all $N$ observations. Note that the critical value and $s_0$ remain fixed from the pilot stage and are not recomputed from the full sample. This is because $N$ was itself determined using $s_0$, so recomputing the standard deviation from all $N$ observations would break the independence between the sample mean and the variance estimate.

**Example**

$d = 10$

$n_0 = 5$

In [3]:
np.random.seed(42)
alpha = 0.05
d = 10
n0 = 5

pilot_sample = np.random.randint(60, 100, size=n0)
s0 = np.std(pilot_sample, ddof=1)
t0 = stats.t.ppf(1 - alpha / 2, df=n0 - 1)
required_sample_size = max(n0, int(np.floor((t0 * s0 / d) ** 2)) + 1)

print(f"Pilot sample: {pilot_sample}")
print(f"Sample standard deviation: {s0:.4f}")
print(f"t-critical: {t0:.4f}")
print(f"Required sample size: {required_sample_size}")


Pilot sample: [98 88 74 67 80]
Sample standard deviation: 12.0748
t-critical: 2.7764
Required sample size: 12


In [4]:
additional_samples = np.random.randint(60, 100, size=required_sample_size - n0)
combined_sample = np.concatenate((pilot_sample, additional_samples))
overall_mean = np.mean(combined_sample)
error_margin = t0 * s0 / np.sqrt(required_sample_size)
confidence_interval = (overall_mean - error_margin, overall_mean + error_margin)

print("Combined sample:", combined_sample)
print(f"Overall mean: {overall_mean:.4f}")
print(f"Error margin: {error_margin:.4f}")
print(f"95% Confidence interval: {confidence_interval}")

Combined sample: [98 88 74 67 80 98 78 82 70 70 83 95]
Overall mean: 81.9167
Error margin: 9.6778
95% Confidence interval: (np.float64(72.23885357798554), np.float64(91.5944797553478))


## Sample Size Selection Based on Statistical Power

As a general guideline, a test is considered to be reliable if it has at least 80% statistical power. To get that power, the real parameter $\theta$ has to be at least $2.8\,SE$ away from the reference point specified in $H_0$.

$H_0: \theta = \theta_0$

$H_1: \theta \neq \theta_0$

The minimum distance required is:

$\theta - \theta_0 = 2.8\,SE$

We know that:

$SE = \dfrac{\sigma}{\sqrt{n}}$

where $\sigma$ is the population standard deviation.

When we solve it for $n$, we get:

$n = \left(\frac{2.8\sigma}{\theta - \theta_0}\right)^2$

$\sigma$ can be obtained from the literature.

**Example**

Suppose we want to test whether a new drug lowers blood pressure. From previous studies, we know that the standard deviation of blood pressure measurements is $\sigma = 15\,\text{mmHg}$. We want our test to be able to detect a difference of at least $\theta - \theta_0 = 5\,\text{mmHg}$ with 80% power.

In [5]:
difference = 5
sigma = 15
n = np.ceil(np.power(2.8 * sigma / difference, 2))
print(f"Required sample size for detecting a difference of {difference} mmHg with sigma={sigma}: {n}")

Required sample size for detecting a difference of 5 mmHg with sigma=15: 71.0
